In [4]:
# SparkSession

from pyspark.sql import SparkSession

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas6_ETL") \
    .master("local[*]") \
    .getOrCreate()

# A. EXTRACT
# 1. Membaca data transaksi (CSV)
df_trx = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)

# 2. Membaca data produk (JSON Lines)
df_produk = spark.read.json("tugas6_produk.json")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 03:33:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/24 03:33:25 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
                                                                                

In [5]:
# A. Extract
# 1. Membaca data transaksi (CSV)
df_trx = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)

# 2. Membaca data produk (JSON Lines)
df_produk = spark.read.json("tugas6_produk.json")

# 3. Membaca data ulasan (CSV)
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

# Menampilkan jumlah baris dan skema masing-masing
print(f"Jumlah baris Transaksi: {df_trx.count()}")
df_trx.printSchema()

print(f"Jumlah baris Produk: {df_produk.count()}")
df_produk.printSchema()

print(f"Jumlah baris Ulasan: {df_ulasan.count()}")
df_ulasan.printSchema()

Jumlah baris Transaksi: 5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Jumlah baris Produk: 30
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Jumlah baris Ulasan: 3500
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [6]:
# B. Transform
from pyspark.sql.functions import col

# 1. Join transaksi ke ulasan (Left Join karena tidak semua transaksi memiliki ulasan)
df_merged = df_trx.join(df_ulasan, on="order_id", how="left")

# 2. Join ke produk (Inner Join karena setiap transaksi pasti memiliki produk yang valid)
df_merged = df_merged.join(df_produk, on="product_id", how="inner")

# 3. Menambahkan kolom total_pendapatan (unit_terjual x harga)
df_merged = df_merged.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))


df_merged.show(5)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
o

In [8]:
# C. Transform

# 1. Menambahkan kolom ada_ulasan SEBELUM mengisi nilai null (seperti instruksi tugas)
df_merged = df_merged.withColumn("ada_ulasan", col("rating").isNotNull())

# 2. Mengisi nilai null pada rating dengan angka 0
df_final = df_merged.na.fill({"rating": 0})

df_final.show(5)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|ada_ulasan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|      true|
|         8|     TX1|           6|2026-10-25 00:00:00|     0|250000|     Fashion|   Produk-8|         1500000|     false|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|      true|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|      true|
|        19|     TX4|           6|2026-10-07 00:00:00|     0| 75000|Rumah Tangga|  Produk-19|          450000|     false|
+----------+--------+---

In [13]:
# D. Load

# Buat folder di HDFS
!hdfs dfs -mkdir -p /user/zerouno/tugas6

# Path dengan skema HDFS eksplisit
path_hdfs = "hdfs://localhost:9000/user/zerouno/tugas6/hasil_etl"

# Simpan data
df_final.write.mode("overwrite").partitionBy("kategori").parquet(path_hdfs)

# Verifikasi struktur HDFS
!hdfs dfs -ls -R /user/zerouno/tugas6/hasil_etl

# Baca kembali data dari HDFS
df_read_back = spark.read.parquet(path_hdfs)
print(f"Jumlah baris tersimpan utuh: {df_read_back.count()}")

-rw-r--r--   3 zerouno supergroup          0 2026-09-24 03:56 /user/zerouno/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - zerouno supergroup          0 2026-09-24 03:56 /user/zerouno/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 zerouno supergroup      16363 2026-09-24 03:56 /user/zerouno/tugas6/hasil_etl/kategori=Elektronik/part-00000-a0dbeb50-6703-43fd-bfec-1a14509158d6.c000.snappy.parquet
drwxr-xr-x   - zerouno supergroup          0 2026-09-24 03:56 /user/zerouno/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 zerouno supergroup      11130 2026-09-24 03:56 /user/zerouno/tugas6/hasil_etl/kategori=Fashion/part-00000-a0dbeb50-6703-43fd-bfec-1a14509158d6.c000.snappy.parquet
drwxr-xr-x   - zerouno supergroup          0 2026-09-24 03:56 /user/zerouno/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 zerouno supergroup      10388 2026-09-24 03:56 /user/zerouno/tugas6/hasil_etl/kategori=Kesehatan/part-00000-a0dbeb50-6703-43fd-bfec-1a14509158d6.c000.snappy.parquet
drwxr-xr-x   - zerouno s

In [14]:
# E. Insight Akhir

from pyspark.sql import functions as F

# Menghitung persentase transaksi dengan ulasan berdasarkan kategori
df_insight = df_final.groupBy("kategori").agg(
    F.count("*").alias("total_transaksi"),
    F.sum(F.when(col("ada_ulasan") == True, 1).otherwise(0)).alias("jumlah_dengan_ulasan")
).withColumn(
    "persentase_ulasan", 
    (col("jumlah_dengan_ulasan") / col("total_transaksi")) * 100
).orderBy("persentase_ulasan")

df_insight.show()

[Stage 52:>                                                         (0 + 1) / 1]

+------------+---------------+--------------------+-----------------+
|    kategori|total_transaksi|jumlah_dengan_ulasan|persentase_ulasan|
+------------+---------------+--------------------+-----------------+
|     Makanan|            536|                 369|68.84328358208955|
|   Kesehatan|            961|                 662|68.88657648283039|
|     Fashion|           1035|                 726|70.14492753623188|
|Rumah Tangga|            815|                 575| 70.5521472392638|
|  Elektronik|           1653|                1168|  70.659407138536|
+------------+---------------+--------------------+-----------------+



**Kategori Transaksi Dengan Ulasan Paling Rendah.**
**Mengapa Hal ini Penting Diketahui oleh Tim Marketing?**

Kategori Makanan memiliki persentase transaksi berulasan paling rendah dibanding kategori lainnya. Hal ini mengindikasikan bahwa pembeli produk makanan cenderung langsung mengonsumsi produk tanpa terdorong untuk memberikan ulasan di platform. Informasi ini penting bagi tim marketing untuk mengevaluasi post-purchase engagement, misalnya dengan memberikan *reward* berupa poin atau voucher diskon khusus untuk mendorong ulasan, mengingat ulasan produk makanan sangat penting untuk membangun kepercayaan calon pembeli baru.